# Exploración de datos — Telco Customer Churn
**Alumno:** Alan Arturo Hernández Chichitz — A01062611

**Objetivo:** entender el dataset y justificar las decisiones de limpieza. El entrenamiento, la evaluación y el registro en MLflow viven en `train.py` (raíz del repo).

**Fuente:** IBM Sample Data Sets vía Kaggle (*Telco Customer Churn*).

 ## 1. Carga y descripción del dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42  # semilla fija para reproducibilidad

RANDOM_STATE es una constante única que se reusa en el split y en el modelo. Así cualquiera que corra el notebook obtiene los mismos resultados (reproducibilidad).

In [ ]:
# Ruta relativa: el notebook está en notebooks/ y el CSV en data/
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(f"Registros: {df.shape[0]:,} | Variables: {df.shape[1]}")
df.head()

In [ ]:
 # Tabla resumen por columna: tipo, cardinalidad, nulos y un valor de ejemplo
resumen = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unicos": df.nunique(),
    "nulos": df.isna().sum(),
    "ejemplo": df.iloc[0],
})
resumen

Mejor que df.info(): n_unicos dice si una variable es categórica (pocos valores) o continua (muchos).

In [ ]:
# TotalCharges debería ser numérica pero pandas la leyó como texto.
# Intentamos convertirla: errors="coerce" transforma en NaN lo que no sea número
tc_numerico = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Filas donde la conversión falló = valores no numéricos escondidos
no_numericos = df.loc[tc_numerico.isna(), "TotalCharges"]
print(f"Valores no convertibles: {len(no_numericos)}")

# repr() muestra el valor "crudo": así se ven los espacios que print() oculta
print(no_numericos.map(repr).value_counts())

In [ ]:
# ¿Qué tienen en común esas filas? Revisamos antigüedad y cargos
df.loc[tc_numerico.isna(), ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

### Diccionario de datos
El dataset tiene **7,043 registros** (uno por cliente) y **21 variables**. Es un problema de **clasificación binaria supervisada**: la variable objetivo es `Churn`.

| Variable | Tipo | Significado |
|---|---|---|
| customerID | Identificador | ID único del cliente (no predictivo) |
| gender | Categórica | Male / Female |
| SeniorCitizen | Binaria (0/1) | 1 si el cliente es adulto mayor |
| Partner | Binaria (Yes/No) | Tiene pareja |
| Dependents | Binaria (Yes/No) | Tiene dependientes |
| tenure | Numérica discreta | Meses que lleva como cliente |
| PhoneService | Binaria | Tiene servicio telefónico |
| MultipleLines | Categórica | Yes / No / No phone service |
| InternetService | Categórica | DSL / Fiber optic / No |
| OnlineSecurity | Categórica | Yes / No / No internet service |
| OnlineBackup | Categórica | Yes / No / No internet service |
| DeviceProtection | Categórica | Yes / No / No internet service |
| TechSupport | Categórica | Yes / No / No internet service |
| StreamingTV | Categórica | Yes / No / No internet service |
| StreamingMovies | Categórica | Yes / No / No internet service |
| Contract | Categórica | Month-to-month / One year / Two year |
| PaperlessBilling | Binaria | Factura electrónica |
| PaymentMethod | Categórica | Electronic check / Mailed check / Bank transfer / Credit card |
| MonthlyCharges | Numérica continua | Cargo mensual actual (USD) |
| TotalCharges | Numérica continua* | Cargo acumulado (USD). *Viene como texto y algunos espacios en blanco, se corrige en el Paso 3 |
| **Churn** | **Binaria (target)** | **Yes si el cliente abandonó en el último mes** |

In [ ]:
 # Balance de clases del target: define qué métricas usar en el Paso 5
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True).round(3))

## Conclusiones
TotalCharges con 11 blancos = clientes con `tenure = 0`; baseline trivial de 73.5%

## 2. Análisis exploratorio (EDA)

In [ ]:
# Copia solo para explorar: TotalCharges a numérico (los 11 blancos quedan como NaN).
# La limpieza "oficial" se hace en el Paso 3; aquí no modificamos df.
df_eda = df.copy()
df_eda["TotalCharges"] = pd.to_numeric(df_eda["TotalCharges"], errors="coerce")
df_eda["churn_bin"] = (df_eda["Churn"] == "Yes").astype(int)  # 1 = abandonó

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
df_eda[num_cols].describe().T

In [ ]:
# Histograma de cada variable numérica separado por Churn.
# stat="density" + common_norm=False: compara FORMAS aunque las clases tengan tamaños distintos (73/27).
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, num_cols):
    sns.histplot(data=df_eda, x=col, hue="Churn", bins=30, stat="density",
                 common_norm=False, element="step", ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot: mediana, rango intercuartil y outliers (puntos fuera de los bigotes) por clase
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(data=df_eda, x="Churn", y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Para cada categoría: % de clientes que abandonaron.
# La media de churn_bin ES la tasa de churn (porque es 0/1).
cat_cols = ["Contract", "InternetService", "PaymentMethod", "TechSupport",
            "OnlineSecurity", "PaperlessBilling", "SeniorCitizen", "gender"]

tasa_global = df_eda["churn_bin"].mean()
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, col in zip(axes.flat, cat_cols):
    tasa = df_eda.groupby(col)["churn_bin"].mean().sort_values(ascending=False)
    sns.barplot(x=tasa.index.astype(str), y=tasa.values, ax=ax)
    ax.axhline(tasa_global, color="red", ls="--", label=f"global {tasa_global:.1%}")  # referencia
    ax.set_title(col)
    ax.set_ylabel("Tasa de churn")
    ax.tick_params(axis="x", rotation=30)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Correlación de Pearson entre numéricas y el target binario.
# Ojo: Pearson solo mide relación LINEAL; sirve como primera aproximación.
corr = df_eda[num_cols + ["churn_bin"]].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlación (Pearson)")
plt.show()

### Hallazgos del EDA
1. Contrato: el factor más fuerte. Mes a mes abandona 42.7% y a dos años solo 2.8%. Negocio: dar incentivos para migrar a contratos anuales.
2. Antigüedad: los primeros 12 meses tienen 47% de churn y más de 48 meses solo 9.5%.
3. Fibra óptica y cargos altos: fibra tiene 41.9% de churn y los que se van pagan 80 USD de mediana contra 64.
4. Servicios de soporte: sin TechSupport ni OnlineSecurity el churn ronda 42%, y con ellos 15%.
5. Pago con cheque electrónico: 45.3% de churn, frente a 15% con pagos automáticos.
6. El género no influye (26.9% vs 26.2%), así que no sirve para predecir.

## 3. Limpieza de datos
Se trabaja sobre una copia `df_clean`; cada decisión queda justificada.

In [ ]:
# Copia de trabajo: el df original se conserva intacto (trazabilidad)
df_clean = df.copy()

# Los 11 valores " " corresponden a clientes con tenure = 0 (Paso 1):
# aún no se les ha facturado nada, así que su cargo acumulado real es 0.
# Imputar 0 es más fiel al negocio que borrar filas o usar la media.
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce").fillna(0)

print(df_clean["TotalCharges"].dtype, "| nulos:", df_clean["TotalCharges"].isna().sum())
print(df_clean.loc[df_clean["tenure"] == 0, ["tenure", "MonthlyCharges", "TotalCharges"]].head())

In [ ]:
# 1) ¿Algún cliente aparece dos veces?
print("customerID repetidos:", df_clean["customerID"].duplicated().sum())

# 2) ¿Filas idénticas en todo excepto el ID?
dup_sin_id = df_clean.drop(columns="customerID").duplicated().sum()
print("Filas idénticas sin contar el ID:", dup_sin_id)

Las 22 son clientes distintos (ID diferente) que casualmente tienen el mismo perfil

In [ ]:
# Regla IQR: es outlier lo que cae fuera de [Q1 - 1.5·IQR, Q3 + 1.5·IQR]
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    q1, q3 = df_clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    fuera = ((df_clean[col] < q1 - 1.5 * iqr) | (df_clean[col] > q3 + 1.5 * iqr)).sum()
    print(f"{col:15s} outliers: {fuera}  | rango: {df_clean[col].min():.1f} – {df_clean[col].max():.1f}")

In [ ]:
# SeniorCitizen viene como 0/1 pero es categórica; la paso a "No"/"Yes"
# para que se trate igual que Partner, Dependents, etc. en el OneHotEncoder
df_clean["SeniorCitizen"] = df_clean["SeniorCitizen"].map({0: "No", 1: "Yes"})

# Target a 0/1: sklearn y las métricas (precision, recall, ROC-AUC) lo esperan numérico
df_clean["Churn"] = (df_clean["Churn"] == "Yes").astype(int)

# customerID es un identificador único: no generaliza, solo haría que el modelo memorice
df_clean = df_clean.drop(columns="customerID")

print(df_clean.shape)
df_clean.dtypes.value_counts()

In [ ]:
# Chequeo de cierre: sin nulos y target con la misma proporción que antes
assert df_clean.isna().sum().sum() == 0, "Quedaron nulos"
print(df_clean["Churn"].value_counts(normalize=True).round(3))
df_clean.head()

### Decisiones de limpieza
| Problema | Decisión | Justificación |
|---|---|---|
| TotalCharges como texto con 11 blancos | Convertir a número e imputar 0 | Son clientes con tenure = 0: aún no se les factura |
| 22 filas idénticas (sin ID) | Conservar | IDs distintos = clientes reales con el mismo perfil |
| Outliers | Ninguno (regla IQR) | Rangos coherentes con el negocio |
| SeniorCitizen 0/1 | Pasar a No/Yes | Es categórica, no una magnitud |
| customerID | Eliminar | Identificador, no aporta para predecir |
| Churn Yes/No | Pasar a 1/0 | Formato requerido por sklearn |

Notas:
1. TotalCharge queda en 0 porque son clientes nuevos sin cobros; la media inventa datos.
2. Para evitar data leakage: lo que aprende de los datos va después del split.
3. Un ID no tiene patrón; el modelo memorizaría en vez de aprender.

## Siguiente paso
Las decisiones de limpieza de la tabla anterior se implementan en `clean_data()` de `train.py`. Aquí se exploran y justifican; el script es la única versión que se ejecuta para entrenar.
